In [49]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
from io import StringIO
import logging

# Configure Chrome options
options = Options()
# options.add_argument('--headless')  # Run in headless mode (no GUI)
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--window-size=1920,1080')

# Set up the driver
from seleniumbase import Driver
from sqlalchemy import create_engine, Column, String, Table, MetaData
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import json
driver= Driver(uc=True, headless=False)


# Define the ProductInfo class
class ProductInfo:
    def __init__(self, product_name, product_price, product_image, sizing_table, product_properties, url):
        self.product_name = product_name
        self.product_price = product_price
        self.product_image = product_image
        self.url = url
        self.sizing_table = pd.read_html(StringIO(sizing_table), header=0, index_col=0)[0] if sizing_table else None
        self.product_properties = product_properties

    def __str__(self):
        return f'Product Name: {self.product_name}, Product Price: {self.product_price}, Product Image: {self.product_image}, Sizing Table: {self.sizing_table}'

    def toJson(self):
        sizes = []
        if self.sizing_table is not None:
            for idx, row in self.sizing_table.T.iterrows():
                parameters = []
                for col in self.sizing_table.T.columns:
                    # if col != 'Größen':
                    parameter_value = str(row[col]).strip()
                    if parameter_value.lower() == 'nan':
                        parameter_value = ""
                    elif '/' in parameter_value:
                        parts = parameter_value.split('/')
                        if all(part.isdigit() for part in parts):
                            parts = sorted(map(int, parts))
                            if parts == list(range(parts[0], parts[-1] + 1)):
                                parameter_value = str((parts[0] + parts[-1]) / 2)
                                # print(f"changed value: {parameter_value} from {parts}")
                                logging.info(f"changed value: {parameter_value} from {parts}")
                            else:
                                # print(f'Warning: {parameter_value} is not a range')
                                logging.warning(f'Warning: {parameter_value} is not a range')
                                parameter_value = None
                        parameter_value = str((parts[0] + parts[-1]) / 2)
                    else:
                        try:
                            parameter_value = str(parameter_value)
                        except ValueError:
                            pass
                    parameters.append({
                        "parameter_name": col,
                        "parameter_value": parameter_value
                    })
                sizes.append({
                    "size_label": idx,
                    "parameters": parameters
                })


        return json.dumps({
            "url": self.url,
            "image_url": self.product_image,
            "price": self.product_price,
            "name": self.product_name,
            "properties": self.product_properties,
            "sizes": sizes
        }, indent=4)
       
driver.get("https://hemden.de")
# Accept cookies
accept_cookies_button = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/a[2]')
accept_cookies_button.click()

In [50]:
url = 'https://www.hemden.de/herrenhemd/olymp/slim-line/bleu/new-kent/kurzarm-12cm/einfarbig/03041211#96969a4f806a8af9d5b12363f06f9329'
def scrapeProductPage(url, driver) -> ProductInfo:
        
    # Load the HTML file
    driver.get(url)


    try:
        # Extract product name
        product_name = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:title"]').get_attribute('content')

        # Extract product price
        product_price = driver.find_element(By.CSS_SELECTOR, 'meta[property="product:price"]').get_attribute('content')

        # Extract product image
        product_image = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute('content')
        # Extract product properties
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.product--properties-list-entry')))
        properties_list = driver.find_elements(By.CSS_SELECTOR, '.product--properties-list-entry')
        print(properties_list)
        product_properties = {}
        for prop in properties_list:
            label = prop.find_element(By.CSS_SELECTOR, '.content--prop-label strong').text.strip(':').strip()
            value = prop.find_element(By.CSS_SELECTOR, '.content--prop-value').text.strip()
            product_properties[label] = value
        # Extract sizing table (if available)
        try:
            table_element = driver.find_element(By.ID, 'product-measures')
            table_html = table_element.get_attribute('outerHTML')  # You can parse this further if needed
        except Exception as e:
            table_html = None  # Handle the case where no table is found

        return ProductInfo(product_name, product_price, product_image, table_html, product_properties, url)
    
    except Exception as e:
        print(f'Error while scraping {url}: {e}')
        return None

In [51]:
product = scrapeProductPage(url, driver)

[<seleniumbase.undetected.webelement.WebElement (session="08016bf0aff2e3af57c5f04190ba5669", element="f.3271433864853DAF1CD14A214747EBF7.d.C035E7F4C67B037F960FFC63A94EDF82.e.131")>, <seleniumbase.undetected.webelement.WebElement (session="08016bf0aff2e3af57c5f04190ba5669", element="f.3271433864853DAF1CD14A214747EBF7.d.C035E7F4C67B037F960FFC63A94EDF82.e.132")>, <seleniumbase.undetected.webelement.WebElement (session="08016bf0aff2e3af57c5f04190ba5669", element="f.3271433864853DAF1CD14A214747EBF7.d.C035E7F4C67B037F960FFC63A94EDF82.e.133")>, <seleniumbase.undetected.webelement.WebElement (session="08016bf0aff2e3af57c5f04190ba5669", element="f.3271433864853DAF1CD14A214747EBF7.d.C035E7F4C67B037F960FFC63A94EDF82.e.134")>, <seleniumbase.undetected.webelement.WebElement (session="08016bf0aff2e3af57c5f04190ba5669", element="f.3271433864853DAF1CD14A214747EBF7.d.C035E7F4C67B037F960FFC63A94EDF82.e.135")>, <seleniumbase.undetected.webelement.WebElement (session="08016bf0aff2e3af57c5f04190ba5669", el

In [52]:
product.product_properties

{'Marke': 'OLYMP',
 'Nachhaltigkeit': 'OLYMP GREENCHOICE',
 'Stoff': 'Chambray',
 'Schnitt': 'Leicht tailliert',
 'Armlänge cm': '12cm',
 'Armlänge': 'Kurzarm',
 'Kragenart': 'New Kent',
 'Material': '100% Baumwolle',
 'Pflege': 'Pflegeleicht',
 'Waschbar bis': '40 Grad',
 'Farbe': 'bleu',
 'Muster': 'Einfarbig',
 'Brusttasche': 'Mit Brusttasche'}

In [53]:
product.toJson()

'{\n    "url": "https://www.hemden.de/herrenhemd/olymp/slim-line/bleu/new-kent/kurzarm-12cm/einfarbig/03041211#96969a4f806a8af9d5b12363f06f9329",\n    "image_url": "https://cdn.hemden.de/media/image/9c/ee/38/0304-12-11_800J5VUPnlKBZRlE.jpg",\n    "price": "59.95",\n    "name": "OLYMP Luxor Modern Fit Hemd bleu, Einfarbig",\n    "properties": {\n        "Marke": "OLYMP",\n        "Nachhaltigkeit": "OLYMP GREENCHOICE",\n        "Stoff": "Chambray",\n        "Schnitt": "Leicht tailliert",\n        "Arml\\u00e4nge cm": "12cm",\n        "Arml\\u00e4nge": "Kurzarm",\n        "Kragenart": "New Kent",\n        "Material": "100% Baumwolle",\n        "Pflege": "Pflegeleicht",\n        "Waschbar bis": "40 Grad",\n        "Farbe": "bleu",\n        "Muster": "Einfarbig",\n        "Brusttasche": "Mit Brusttasche"\n    },\n    "sizes": [\n        {\n            "size_label": "XS",\n            "parameters": [\n                {\n                    "parameter_name": "Kragenweite",\n                  

In [54]:
import json
with open('product.json', 'w') as f:
    f.write(product.toJson())

In [55]:
import requests

url = "http://localhost:8000/products/"

payload = product.toJson()

with open ('product.json', 'r') as f:
    payload = json.load(f)
    
headers = {
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response.status_code)
print(response.json())

200
{'url': 'https://www.hemden.de/herrenhemd/olymp/slim-line/bleu/new-kent/kurzarm-12cm/einfarbig/03041211#96969a4f806a8af9d5b12363f06f9329', 'price': '59.95', 'properties': {'Marke': 'OLYMP', 'Nachhaltigkeit': 'OLYMP GREENCHOICE', 'Stoff': 'Chambray', 'Schnitt': 'Leicht tailliert', 'Armlänge cm': '12cm', 'Armlänge': 'Kurzarm', 'Kragenart': 'New Kent', 'Material': '100% Baumwolle', 'Pflege': 'Pflegeleicht', 'Waschbar bis': '40 Grad', 'Farbe': 'bleu', 'Muster': 'Einfarbig', 'Brusttasche': 'Mit Brusttasche'}, 'id': 10, 'image_url': 'https://cdn.hemden.de/media/image/9c/ee/38/0304-12-11_800J5VUPnlKBZRlE.jpg', 'name': 'OLYMP Luxor Modern Fit Hemd bleu, Einfarbig'}


In [56]:
driver.quit()